# PS 3 — Titanic Passenger Dataset (Seaborn)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** detect missing data, impute numeric (median) and categorical (mode), standardize strings, ordinal-encode ticket class, one-hot encode nominal predictors, and normalize numeric predictors.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA/'titanic.csv')
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

**Note:** `survived` is the target. Do not scale it. `pclass` is ordered 1st/2nd/3rd, while `sex`, `embarked`, etc. are nominal. Seaborn columns such as `alive`, `class`, `who`, `adult_male` can duplicate or leak target information; this notebook processes a practical subset and retains `survived` unchanged.

In [ ]:
keep = ['survived','pclass','sex','age','sibsp','parch','fare','embarked','alone']
df = df[keep].copy()
df = df.drop_duplicates().copy()
num_cols = ['age','sibsp','parch','fare']
nominal_cols = ['sex','embarked']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df[c] = df[c].fillna(df[c].median())
for c in nominal_cols:
    df[c] = df[c].astype('string').str.strip().str.lower().replace('',pd.NA)
    df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])
df['pclass'] = pd.to_numeric(df['pclass'],errors='coerce').fillna(df['pclass'].mode()[0]).astype(int)
df['alone'] = df['alone'].astype(int)
df['survived'] = pd.to_numeric(df['survived'],errors='coerce').astype(int)
print('After imputation:');print(df.isna().sum())
display(df.head())

**Encoding:** `pclass` is already an ordinal 1/2/3 code; `alone` is Boolean → 0/1; use one-hot on nominal `sex` and `embarked`. Min-Max normalize age, fare, siblings and parents/children counts.

In [ ]:
print('Before scaling:'); display(df[num_cols].head())
df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])
print('After scaling:'); display(df[num_cols].head())
df = pd.get_dummies(df,columns=nominal_cols,dtype=int)

In [ ]:
print('BEFORE preprocessing:'); display(before.head())
print('AFTER preprocessing:'); display(df.head())
print('Final shape:', df.shape)
print('Remaining missing values:', df.isna().sum().sum())
print('Remaining duplicates:', df.duplicated().sum())

In [ ]:
df.to_csv(OUT/'PS03_titanic_processed.csv',index=False)